# 🚀 Phase 11 Training Notebook

## Context:
- **Phase 10** consolidated the label fixes and ran initial fine-tuning.
- **Phase 11 Goal**: Resume from Phase 10 checkpoints to refine the model.
- **Strategy**: Load Phase 10 weights, verify dataset counts, and fine-tune with a low learning rate.

## Phase 11 Configuration:
- **Base Model**: Phase 10 Checkpoints (Medium Tier)
- **LR: 1e-5** (Very low for delicate fine-tuning)
- **Epochs: 10**
- **Data**: `phase11_labels.csv`


## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone/Update Repository

In [ ]:
import os

if not os.path.exists('/content/phase2'):
    !git clone https://github.com/nithin12342/phase2.git /content/phase2
    print("✅ Repository cloned successfully!")
else:
    %cd /content/phase2
    !git fetch origin
    !git reset --hard origin/main
    %cd /content
    print("✅ Repository updated to latest!")

## Step 3: Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers h5py pandas scikit-learn tqdm --quiet
print("✅ Dependencies installed!")

## Step 4: Configure Paths & Hyperparameters

In [ ]:
# ============================================
# Phase 11 Configuration
# ============================================

OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
DATA_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output"
LABELS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/all_labels.csv"

# Input (Phase 10) and Output (Phase 11) directories
PHASE10_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase10"
OUT_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase11"

# Fine-tuning hyperparameters
TIER = "medium"
EPOCHS = 10
LR = 1e-5

import os

!mkdir -p {OUT_DIR}
print(f"📁 Output Directory: {OUT_DIR}")

print(f"\n📁 OS_PATH exists: {os.path.exists(OS_PATH)}")
print(f"📁 DATA_DIR exists: {os.path.exists(DATA_DIR)}")
print(f"📁 LABELS exists: {os.path.exists(LABELS)}")
print(f"📁 PHASE10_DIR exists: {os.path.exists(PHASE10_DIR)}")
print(f"\n⚙️ Phase 11 Configuration:")
print(f"   Tier: {TIER}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning Rate: {LR}")

## Step 5: Filter & Verify Data

In [ ]:
import pandas as pd

TARGET_SOURCES = ['DAIC-WOZ', 'Extended-DAIC', 'EATD-Corpus']
PHASE11_LABELS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/phase11_labels.csv"

if os.path.exists(LABELS):
    df = pd.read_csv(LABELS)
    print(f"📊 Original Samples: {len(df)}")
    
    if 'Source' in df.columns:
        df_filtered = df[df['Source'].isin(TARGET_SOURCES)].copy()
        print(f"📉 Filtered Samples: {len(df_filtered)}")
        
        df_filtered.to_csv(PHASE11_LABELS, index=False)
        LABELS = PHASE11_LABELS
        print(f"✅ Saved Phase 11 Labels to: {LABELS}")
        
        dist = df_filtered['Source'].value_counts()
        print("\n🌍 Source Distribution:")
        print(dist)
    else:
        print("⚠️ 'Source' column missing. Using all labels.")
    
    print("\n⚖️ Class Distribution:")
    print(df_filtered['Depression_Label'].value_counts(normalize=True))
else:
    print("❌ Labels CSV not found!")

## Step 6: Fine-Tune All 5 Folds

In [ ]:
import time
import glob

total_start = time.time()

for fold in range(5):
    print(f"\n{'='*60}")
    print(f"🚀 PHASE 11 - FOLD {fold}/4")
    print(f"{'='*60}\n")

    ckpt_pattern = f"{PHASE10_DIR}/h5_omnifusion_{TIER}_fold{fold}_best.pt"
    checkpoints = glob.glob(ckpt_pattern)
    
    resume_arg = ""
    if checkpoints:
        resume_path = checkpoints[0]
        print(f"🔄 RESUMING from Phase 10: {os.path.basename(resume_path)}")
        resume_arg = f"--resume {resume_path}"
    else:
        print(f"⚠️ No Phase 10 checkpoint found for fold {fold}. Checking Phase 9...")
        ckpt_pattern_9 = f"/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase9/h5_omnifusion_{TIER}_fold{fold}_best.pt"
        checkpoints_9 = glob.glob(ckpt_pattern_9)
        if checkpoints_9:
             resume_path = checkpoints_9[0]
             print(f"🔄 RESUMING from Phase 9: {os.path.basename(resume_path)}")
             resume_arg = f"--resume {resume_path}"
        else:
             print("⚠️ No checkpoint found. Starting fresh.")

    fold_start = time.time()

    !PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/train.py \
        --data_dir {DATA_DIR} \
        --labels_csv {LABELS} \
        --output_dir {OUT_DIR} \
        --tier {TIER} \
        --epochs {EPOCHS} \
        --fold_idx {fold} \
        --lr {LR} \
        {resume_arg}

    fold_time = time.time() - fold_start
    print(f"\n⏱️ Fold {fold} completed in {fold_time/60:.1f} minutes")

total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"✅ ALL 5 FOLDS COMPLETED!")
print(f"⏱️ Total training time: {total_time/60:.1f} minutes")
print(f"{'='*60}")

## Step 7: Verify Checkpoints

In [ ]:
import os
import glob

print("📦 Phase 11 Checkpoints:")
checkpoints = sorted(glob.glob(f"{OUT_DIR}/*_best.pt"))

if not checkpoints:
    print("   ⚠️ No checkpoints found!")
else:
    for ckpt in checkpoints:
        size_mb = os.path.getsize(ckpt) / (1024*1024)
        print(f"   ✅ {os.path.basename(ckpt)} ({size_mb:.1f} MB)")
    print(f"\n✅ Total: {len(checkpoints)} checkpoints")

## Step 8: Run Ensemble Prediction

In [ ]:
print("🔮 Generating predictions...\n")

!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/ensemble_predict.py \
    --checkpoints {OUT_DIR} \
    --input {DATA_DIR} \
    --tier {TIER} \
    --output "/content/phase11_results.csv"

import os
if os.path.exists("/content/phase11_results.csv"):
    print("\n✅ Ensemble predictions saved!")
else:
    print("\n❌ Prediction failed!")

## Step 9: Evaluate Results

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix

print("🏆 Results Analysis...")

results_df = pd.read_csv("/content/phase11_results.csv")
labels_df = pd.read_csv(LABELS)

results_df['pid'] = results_df['pid'].astype(str)
labels_df['Participant_ID'] = labels_df['Participant_ID'].astype(str)
merged = results_df.merge(labels_df, left_on='pid', right_on='Participant_ID', how='inner')

y_prob = merged['probability'].values
if 'PHQ8_Binary' in merged.columns:
    y_true = merged['PHQ8_Binary'].values
else:
    y_true = (merged['PHQ8_Score'] >= 10).astype(int).values

best_f1, best_thresh = 0, 0.45
for thresh in np.arange(0.30, 0.70, 0.01):
    y_pred = (y_prob >= thresh).astype(int)
    f1 = f1_score(y_true, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

y_pred_opt = (y_prob >= best_thresh).astype(int)
f1 = f1_score(y_true, y_pred_opt)
precision = precision_score(y_true, y_pred_opt)
recall = recall_score(y_true, y_pred_opt)
accuracy = accuracy_score(y_true, y_pred_opt)
cm = confusion_matrix(y_true, y_pred_opt)

print(f"\n{'='*50}")
print(f"📊 FINAL RESULTS (Threshold={best_thresh:.2f})")
print(f"{'='*50}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Confusion Matrix:")
print(cm)
print(f"{'='*50}")

print("\n🎯 Target Check (≥ 0.88):")
print(f"  {'✅' if f1 >= 0.88 else '❌'} F1: {f1:.4f}")
print(f"  {'✅' if recall >= 0.88 else '❌'} Recall: {recall:.4f}")
print(f"  {'✅' if precision >= 0.88 else '❌'} Precision: {precision:.4f}")

## Step 10: Save Results

In [ ]:
!cp /content/phase11_results.csv "/content/drive/MyDrive/DAIC-WOZ_Datasets/phase11_final_results.csv"
print("✅ Results saved to Google Drive!")

print(f"\n{'='*50}")
print(f"🏆 PHASE 11 COMPLETE")
print(f"{'='*50}")
print(f"📁 Checkpoints: {OUT_DIR}")
print(f"🎯 Best F1: {best_f1:.4f}")
print(f"{'='*50}")